# Homework 2

This Notebook will detail Homework 2, which involves a basic capacity expansion model formulation described in [Notebook 3](https://github.com/east-winds/power-systems-optimization/tree/master/Notebooks)

First, load (or install if necessary) a set of packages you'll need for this assignment...

In [1]:
using JuMP
using HiGHS
using DataFrames
using CSV

### Question 1 - Build the basic thermal generation expansion model

Using the example model in [Notebook 3](https://github.com/east-winds/power-systems-optimization/tree/master/Notebooks) as your guide, input the code to create a basic thermal generator capacity expansion model, including [downloading the data for Notebook 3 here](https://github.com/east-winds/power-systems-optimization/tree/master/Notebooks/expansion_data) and loading the appropriate csv files.

In [2]:
# Load parameters
path = joinpath(dirname(pwd()),"Notebooks","expansion_data")
generators = DataFrame(CSV.File(joinpath(path,"generators_for_expansion.csv")))
demand = DataFrame(CSV.File(joinpath(path,"demand_for_expansion.csv")))
NSECost = 9000

# Define sets
G_new_thermal = generators.G[1:4]
H = demand.Hour;

In [3]:
# Create the model
Thermal_Expansion_Model = Model(HiGHS.Optimizer)

@variables(Thermal_Expansion_Model, begin
    CAP[g in G_new_thermal] >=0
    GEN[g in G_new_thermal, h in H] >= 0
    NSE[h in H] >= 0
end);

@constraints(Thermal_Expansion_Model, begin
    cDemandBalance[h in H], sum(GEN[g,h] for g in G_new_thermal) + NSE[h] == demand.Demand[h]
    cCapacity[g in G_new_thermal, h in H], GEN[g,h] <= CAP[g]
end);

@objective(Thermal_Expansion_Model, Min,
    sum(generators.FixedCost[generators.G.==g][1]*CAP[g] + 
        sum(generators.VarCost[generators.G.==g][1]*GEN[g,h] for h in H)
        for g in G_new_thermal) + 
    sum(NSECost*NSE[h] for h in H) 
);

optimize!(Thermal_Expansion_Model);

Running HiGHS 1.11.0 (git hash: 364c83a51e): Copyright (c) 2025 HiGHS under MIT licence terms
LP   has 43800 rows; 43804 cols; 113880 nonzeros
Coefficient ranges:
  Matrix [1e+00, 1e+00]
  Cost   [2e+01, 6e+05]
  Bound  [0e+00, 0e+00]
  RHS    [1e+03, 5e+03]
Presolving model
43800 rows, 43804 cols, 113880 nonzeros  0s
Dependent equations search running on 8760 equations with time limit of 1000.00s
Dependent equations search removed 0 rows and 0 nonzeros in 0.00s (limit = 1000.00s)
43800 rows, 43804 cols, 113880 nonzeros  0s
Presolve : Reductions: rows 43800(-0); columns 43804(-0); elements 113880(-0) - Not reduced
Problem not reduced by presolve: solving the LP
Using EKK dual simplex solver - serial
  Iteration        Objective     Infeasibilities num(sum)
          0     0.0000000000e+00 Pr: 8760(2.25679e+07) 0s
      27493     9.9087397880e+08 Pr: 0(0) 0s
Model status        : Optimal
Simplex   iterations: 27493
Objective value     :  9.9087397880e+08
P-D objective error :  3.3144637

In [4]:
# Format capacity and generation results in table
generation = zeros(size(G_new_thermal,1))
for i in 1:size(G_new_thermal,1) 
    generation[i] = sum(value.(GEN)[G_new_thermal[i],:].data) 
end
MWh_share = generation./sum(demand.Demand).*100
results = DataFrame(
    Resource = G_new_thermal, 
    MW = value.(CAP).data,
    GWh = generation/1000,
    Percent_GWh = MWh_share
)
NSE_MW = maximum(value.(NSE).data) 
NSE_MWh = sum(value.(NSE).data)
push!(results, ["NSE" NSE_MW NSE_MWh/1000 NSE_MWh/sum(demand.Demand)*100])

Row,Resource,MW,GWh,Percent_GWh
,String7,Float64,Float64,Float64
1,Geo,0.0,-0.0,-0.0
2,Coal,0.0,-0.0,-0.0
3,CCGT,3328.0,22139.4,98.1011
4,CT,1290.0,427.894,1.89603
5,NSE,195.0,0.637,0.00282259


## Question 2: Analytical solution

**A.** Using the data provided above, sort the demand data from highest to lowest hours to create a load duration curve and save this as a vector/array/DataFrame of your choice.

In [5]:
LDC_vector = sort(demand.Demand, rev=true);

**B.** Now using the cost data provided in '/generators_for_expansion.csv' and the load duration curve above, use the formulas provided in Lecture to determine an analytical solution to the optimal thermal generation expansion decisions (e.g. solve it algebraically rather than use an optimization solver to find the solution). 

Report the optimal capacity of each generation source and compare to the solution from the optimization model above. 

Show your work in cells below, using Julia to perform calculations. Explain your steps using inline code comments (e.g. `# Comment`) or by interspersing Markdown cells.  

Tip: round your solutions for the crossover hour between each technology to the nearest integer (as we have discrete hours in the time series).

In [6]:
# Find lower hull of crossover hours (and compute corresponding capacity) for all total cost functions between h=0 and h=8760

# Add NSE data to new thermal generators df and sort by ascending fixed cost (y intercepts) so searching for least cost crossovers is quicker
cost_data = sort(push!(generators[1:4,:], ["NSE" "NSE" 0 0 0 0 0 0 0 0 0 0 NSECost]),:FixedCost)

# Initialize search at NSE (corresponding to row i = 1 in the cost_data df) and first hour (according to LDC vector)
i = 1
h = 1

while h <= 8760  # Stop when technology option is not optimal for any hours in the year
    c = Inf # Initialize upper bounds for cost search
    prev_h = h # Update crossover hour as lower bound for search
    temp_i = i # Update technology for search
    for j in 1:5
        if j != i # Skip checking crossover with current technology
            m1, b1 = cost_data.VarCost[i], cost_data.FixedCost[i]
            m2, b2 = cost_data.VarCost[j], cost_data.FixedCost[j]
            if m1 != m2  # Skip parallel lines
                temp_h = (b2 - b1) / (m1 - m2)  # Compute crossover hour
                temp_c = m1 * temp_h + b1  # Compute cost at crossover hour
                if temp_c < c && temp_h > prev_h # Store lowest cost crossover point
                    c = temp_c
                    h = temp_h
                    temp_i = j
                end
            end
        end
    end
    if h <= 8760 # print optimal capacity (continue search if there exists an additional optimal technology within the year; break if not)
        println("$(cost_data.G[temp_i-1]) Capacity = $(LDC_vector[ceil(Int, prev_h)] - LDC_vector[ceil(Int, h)]) MW")
        i = temp_i
        if (LDC_vector[ceil(Int, prev_h)] - LDC_vector[ceil(Int, h)]) == results[findfirst(isequal(cost_data.G[temp_i-1]), results[!, :Resource]),:MW][1] # check against optimization model result
            println("The optimal capacity solved algebraically is the same as the solution from the optimization model.")
        end
    else
        println("$(cost_data.G[i]) Capacity = $(LDC_vector[ceil(Int, prev_h)]) MW")
        if (LDC_vector[ceil(Int, prev_h)]) == results[findfirst(isequal(cost_data.G[i]), results[!, :Resource]),:MW][1] # check against optimization model result
            println("The optimal capacity solved algebraically is the same as the solution from the optimization model.")
        end
        break
    end
end

NSE Capacity = 195 MW
The optimal capacity solved algebraically is the same as the solution from the optimization model.
CT Capacity = 1290 MW
The optimal capacity solved algebraically is the same as the solution from the optimization model.
CCGT Capacity = 3328 MW
The optimal capacity solved algebraically is the same as the solution from the optimization model.


**C.** Now change the fuel cost of natural gas to \$8.00/MMBtu, recalculate the variable cost of CCGTs and CTs, and solve again for the optimal generation capacity mix. Describe what changes in your capacity results and what doesn't, and provide an explanation.

In [7]:
for g in ["CCGT", "CT"]
    generators.FuelCost[generators.G .== g] .= 8.0
    generators.VarCost[generators.G .== g] .= generators.VarOM[generators.G .== g] .+ generators.HeatRate[generators.G .== g] .* generators.FuelCost[generators.G .== g]
end

@objective(Thermal_Expansion_Model, Min,
    sum(generators[generators.G.==g,:FixedCost][1]*CAP[g] + 
    sum(generators[generators.G.==g,:VarCost][1]*GEN[g,h] for h in H)
    for g in G_new_thermal) + 
    sum(NSECost*NSE[h] for h in H) 
);

optimize!(Thermal_Expansion_Model);

LP   has 43800 rows; 43804 cols; 113880 nonzeros
Coefficient ranges:
  Matrix [1e+00, 1e+00]
  Cost   [2e+01, 6e+05]
  Bound  [0e+00, 0e+00]
  RHS    [1e+03, 5e+03]
Solving LP without presolve, or with basis, or unconstrained
Using EKK dual simplex solver - serial
  Iteration        Objective     Infeasibilities num(sum)
          0    -1.1373578747e+05 Ph1: 25077(50154); Du: 4(113736) 0s
       7273     1.4540758470e+09 Pr: 0(0) 2s
Model status        : Optimal
Simplex   iterations: 7273
Objective value     :  1.4540758470e+09
P-D objective error :  1.5904673895e-14
HiGHS run time      :          1.75


In [8]:
# Format capacity and generation results in table
generation = zeros(size(G_new_thermal,1))
for i in 1:size(G_new_thermal,1) 
    generation[i] = sum(value.(GEN)[G_new_thermal[i],:].data) 
end
MWh_share = generation./sum(demand.Demand).*100
results = DataFrame(
    Resource = G_new_thermal, 
    MW = value.(CAP).data,
    GWh = generation/1000,
    Percent_GWh = MWh_share
)
NSE_MW = maximum(value.(NSE).data) 
NSE_MWh = sum(value.(NSE).data)
push!(results, ["NSE" NSE_MW NSE_MWh/1000 NSE_MWh/sum(demand.Demand)*100])

Row,Resource,MW,GWh,Percent_GWh
,String7,Float64,Float64,Float64
1,Geo,0.0,-0.0,-0.0
2,Coal,2110.0,17751.1,78.6565
3,CCGT,1472.0,4618.47,20.4648
4,CT,1036.0,197.67,0.87589
5,NSE,195.0,0.637,0.00282259


With the increased fuel cost of natural gas, some natural gas capacity (especially for load-following CCGT) in the prior result is replaced by coal. CT capacity decreases by 254 MW (from 1,290 MW to 1,036 MW), CCGT capacity decreases by 1,856 MW (from 3,328 to 1,472 MW), coal capacity increases by 2110 MW (from 0 MW; totalling the decrease in natural gas capacity), and NSE capacity remains unchanged.

An increase in fuel cost increases the unit's total variable cost, i.e. increases the slope of its cost functions. This leads to lower crossover hours between CT and CCGT (therefore has lower CT capacity) as well as a crossover hour between CCGT and coal that is now below 8760 (therefore has lower CCGT capacity and existence of coal capacity), meaning coal is also economic to operate within the year. NSE capacity remains unchanged because the crossover hour only changed slightly (since NSE cost function also remains unchanged) and still occurs between the same hour interval 7:8.

## Question 3 - Expansion with renewables

**A.** Using JuMP/Julia, implement an optimization model based on the formulation for optimal thermal+renewable capacity expansion provided in Section 2 of [Notebook 3](https://github.com/east-winds/power-systems-optimization/tree/master/Notebooks). 

In [9]:
generators = DataFrame(CSV.File(joinpath(path,"generators_for_expansion.csv")))
variability = DataFrame(CSV.File(joinpath(path,"wind_solar_for_expansion.csv")))
G_new = generators.G[:]
G_new_RE = generators.G[end-1:end]

# Create the model
RE_Expansion_Model = Model(HiGHS.Optimizer)

@variables(RE_Expansion_Model, begin
        CAP[g in G_new] >=0
        GEN[g in G_new, h in H] >= 0
        NSE[h in H] >= 0
end);

@constraints(RE_Expansion_Model, begin
    cDemandBalance[h in H], sum(GEN[g,h] for g in G_new) + NSE[h] == demand.Demand[h]
    cCapacity_new_thermal[g in G_new_thermal, h in H], GEN[g,h] <= CAP[g]
    cCapacity_new_RE[g in G_new_RE, h in H], GEN[g,h] <= CAP[g] * variability[h,g] # Additional constraint for RE dependent on CFs
end);

@objective(RE_Expansion_Model, Min,
    sum(generators.FixedCost[generators.G.==g][1]*CAP[g] + 
        sum(generators.VarCost[generators.G.==g][1]*GEN[g,h] for h in H)
        for g in G_new) + 
    sum(NSECost*NSE[h] for h in H) 
);

**B.** Solve the model to determine the optimal capacity when wind and solar are available resources and extract results for generation and capacity.

In [10]:
optimize!(RE_Expansion_Model);

Running HiGHS 1.11.0 (git hash: 364c83a51e): Copyright (c) 2025 HiGHS under MIT licence terms
LP   has 61320 rows; 61326 cols; 161976 nonzeros
Coefficient ranges:
  Matrix [5e-05, 1e+00]
  Cost   [2e+01, 6e+05]
  Bound  [0e+00, 0e+00]
  RHS    [1e+03, 5e+03]
Presolving model
56856 rows, 56862 cols, 153048 nonzeros  0s
Dependent equations search running on 8760 equations with time limit of 1000.00s
Dependent equations search removed 0 rows and 0 nonzeros in 0.00s (limit = 1000.00s)
56856 rows, 56862 cols, 153048 nonzeros  0s
Presolve : Reductions: rows 56856(-4464); columns 56862(-4464); elements 153048(-8928)
Solving the presolved LP
Using EKK dual simplex solver - serial
  Iteration        Objective     Infeasibilities num(sum)
          0     0.0000000000e+00 Pr: 8760(6.18517e+06) 0s
      40370     8.2899893531e+08 Pr: 0(0); Du: 0(3.09303e-11) 3s
Solving the original LP from the solution after postsolve
Model status        : Optimal
Simplex   iterations: 40370
Objective value     : 

In [11]:
# Format capacity and generation results in table
generation = zeros(size(G_new,1))
for i in 1:size(G_new,1) 
    generation[i] = sum(value.(GEN)[G_new[i],:].data) 
end
MWh_share = generation./sum(demand.Demand).*100
results = DataFrame(
    Resource = G_new, 
    MW = value.(CAP).data,
    GWh = generation/1000,
    Percent_GWh = MWh_share
)
NSE_MW = maximum(value.(NSE).data) 
NSE_MWh = sum(value.(NSE).data)
push!(results, ["NSE" NSE_MW NSE_MWh/1000 NSE_MWh/sum(demand.Demand)*100])

Row,Resource,MW,GWh,Percent_GWh
,String7,Float64,Float64,Float64
1,Geo,0.0,0.0,0.0
2,Coal,0.0,0.0,0.0
3,CCGT,2422.86,11367.7,50.3709
4,CT,1419.78,453.98,2.01162
5,Wind,349.956,1007.7,4.46518
6,Solar,3362.77,9738.16,43.1505
7,NSE,136.513,0.399328,0.00176945


**C.** What happens to the total firm generation and maximum MW of non-served energy? What does this imply about the capacity value of solar and/or wind built in the optimal capacity mix?

Total firm generation and maximum MW of non-served energy both decrease (by 775.36 MW and 58.487 MW respectively). This implies that even with significantly less (~16.8%) firm generation, the reliability of the system increases (since NSE is inversely related to reliability) because solar and wind provide some capacity value to the system. Although variable (MW availability scaled by capacity factor), solar and wind provide generation during peak hours.

## Question 4: Brownfield Expansion Model

**A.** Now implement an optimization model based on the formulation for optimal "brownfield" thermal+renewable capacity expansion (e.g. with existing generators) provided in Section 3 of [Notebook 3](https://github.com/east-winds/power-systems-optimization/tree/master/Notebooks).

Use the following data for fixed and variable costs of existing gas capacity. Note: unlike in the formulation in Notebook 3, there is no existing renewable capacity here to consider (only thermal).

In [12]:
generators = DataFrame(CSV.File(joinpath(path,"generators_for_expansion.csv")))

# Add parameters for existing CCGTs, with the set index "Old"
push!(generators, ["Old_CC" "Existing CCGT" 0 40000 5 7.5 4 0 0 0 0 40000 30])
# Add parameters for existing CTs, with the set index "Old"
push!(generators, ["Old_CT" "Existing CT" 0 30000 11 11.0 4 0 0 0 0 30000 55])

# Set installed capacity for existing CCGTs:
ExistingCap_CCGT = 1260 # Approximate actual existing capacity in SDGE
ExistingCap_CT = 925 # Approximate actual existing capacity in SDGE
# Add new column to generators Data Frame
generators[!,:ExistingCap] = [0,0,0,0,0,0, ExistingCap_CCGT, ExistingCap_CT];

In [13]:
G_all = generators.G[:]
G_old_thermal = generators.G[end-1:end]

# Create the model
Brownfield_Expansion_Model = Model(HiGHS.Optimizer)

@variables(Brownfield_Expansion_Model, begin
        CAP[g in G_new] >=0
        GEN[g in G_all, h in H] >= 0
        NSE[h in H] >= 0
        RET[g in G_old_thermal] >= 0     # Additional variable to represent capacity retired (MW)
end);

@constraints(Brownfield_Expansion_Model, begin
    cDemandBalance[h in H], sum(GEN[g,h] for g in G_all) + NSE[h] == demand.Demand[h]
    cCapacity_new_thermal[g in G_new_thermal, h in H], GEN[g,h] <= CAP[g]
    cCapacity_old_thermal[g in G_old_thermal, h in H], GEN[g,h] <= generators[generators.G.==g,:ExistingCap][1] - RET[g] # Additional constraint on existing thermal capacity
    cCapacity_new_RE[g in G_new_RE, h in H], GEN[g,h] <= CAP[g] * variability[h,g]
end);

@objective(Brownfield_Expansion_Model, Min,
    sum(generators.FixedCost[generators.G.==g][1]*CAP[g] for g in G_new) + # New generators fixed cost
    sum(generators.FixedCost[generators.G.==g][1]*(generators.ExistingCap[generators.G.==g][1]-RET[g]) for g in G_old_thermal) + # Existing generators fixed cost
    sum(sum(generators.VarCost[generators.G.==g][1]*GEN[g,h] for h in H) for g in G_all) + 
    sum(NSECost*NSE[h] for h in H) 
);

**B.** Solve the model to determine the optimal capacity when with existing generators and extract results for generation and capacity (including retirements).

In [ ]:
optimize!(Brownfield_Expansion_Model);

Running HiGHS 1.11.0 (git hash: 364c83a51e): Copyright (c) 2025 HiGHS under MIT licence terms
LP   has 78840 rows; 78848 cols; 214536 nonzeros
Coefficient ranges:
  Matrix [5e-05, 1e+00]
  Cost   [2e+01, 6e+05]
  Bound  [0e+00, 0e+00]
  RHS    [9e+02, 5e+03]
Presolving model
74376 rows, 74384 cols, 205608 nonzeros  0s
Dependent equations search running on 8760 equations with time limit of 1000.00s
Dependent equations search removed 0 rows and 0 nonzeros in 0.00s (limit = 1000.00s)
74376 rows, 74384 cols, 205608 nonzeros  0s
Presolve : Reductions: rows 74376(-4464); columns 74384(-4464); elements 205608(-8928)
Solving the presolved LP
Using EKK dual simplex solver - serial
  Iteration        Objective     Infeasibilities num(sum)
          0    -6.9996449302e+04 Ph1: 17520(10432); Du: 2(69996.4) 0s
      45746     7.5446152159e+08 Pr: 3007(1.93685e+06); Du: 0(7.59295e-08) 5s
      46776     7.5494110973e+08 Pr: 0(0); Du: 0(5.19466e-11) 6s
Solving the original LP from the solution after 

In [ ]:
# Format capacity and generation results in table
generation = zeros(size(G_all,1))
for i in 1:size(G_all,1) 
    generation[i] = sum(value.(GEN)[G_all[i],:].data) 
end
MWh_share = generation./sum(demand.Demand).*100
capacity = push!(value.(CAP).data, generators[generators.G.=="Old_CC",:ExistingCap][1]-value.(RET).data[1], generators[generators.G.=="Old_CT",:ExistingCap][1]-value.(RET).data[2])
results = DataFrame(
    Resource = G_all, 
    MW = capacity,
    Retired_MW = push!(zeros(size(G_new,1)),value.(RET).data[1],value.(RET).data[2]),
    GWh = generation/1000,
    Percent_GWh = MWh_share
)
NSE_MW = maximum(value.(NSE).data) 
NSE_MWh = sum(value.(NSE).data)
push!(results, ["NSE" NSE_MW 0 NSE_MWh/1000 NSE_MWh/sum(demand.Demand)*100])

Row,Resource,MW,Retired_MW,GWh,Percent_GWh
,String7,Float64,Float64,Float64,Float64
1,Geo,0.0,0.0,0.0,0.0
2,Coal,0.0,0.0,0.0,0.0
3,CCGT,1417.31,0.0,8545.01,37.8636
4,CT,238.555,0.0,105.948,0.469465
5,Wind,385.045,0.0,1111.73,4.92614
6,Solar,3357.62,0.0,9717.83,43.0604
7,Old_CC,1260.0,0.0,2973.4,13.1754
8,Old_CT,925.0,0.0,113.611,0.503418
9,NSE,125.066,0.0,0.366094,0.00162219
